# Aisle pacing ratio: with vs. without `user_aisle_pacing_ratio`
Trains the XGBoost classifier twice on a small user sample, identical settings, differing only by the new feature.
Compares (1) feature importance (gain and SHAP) and (2) validation average precision (PR-AUC, the classifier's training metric).

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import source_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import split_data, select_features, get_bq_data_sample

import warnings
warnings.filterwarnings('ignore')

##### Read the 10% user sample and thin it to a light 20% of those users

In [ ]:
df_sample = get_bq_data_sample()

light_users = pd.Series(df_sample['user_id'].unique()).sample(frac=0.20, random_state=42)
df_light = df_sample[df_sample['user_id'].isin(light_users)]

df_features, feature_list, sort_list = select_features(df_light)
train_df, val_df, _ = split_data(df_features)

assert 'user_aisle_pacing_ratio' in feature_list
assert set(train_df['user_id']).isdisjoint(val_df['user_id'])
print(f"Train rows: {len(train_df)}, validation rows: {len(val_df)}")

##### Train with and without the new feature (same fixed hyperparameters, early stopping on validation PR-AUC)

In [ ]:
def train_and_score(features):
    model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', tree_method='hist', device='cuda',
                              learning_rate=0.05, n_estimators=2000, max_depth=7, colsample_bytree=0.8,
                              reg_lambda=1.0, gamma=0.1, early_stopping_rounds=50, random_state=42)
    model.fit(train_df[features], train_df['label_reordered'],
              eval_set=[(val_df[features], val_df['label_reordered'])], verbose=False)
    val_aucpr = max(model.evals_result()['validation_0']['aucpr'])
    return model, val_aucpr

features_without = [f for f in feature_list if f != 'user_aisle_pacing_ratio']

model_without, aucpr_without = train_and_score(features_without)
model_with, aucpr_with = train_and_score(feature_list)

##### Main evaluation metric: validation average precision (PR-AUC)

In [ ]:
metric_comparison = pd.DataFrame({
    'model': ['without user_aisle_pacing_ratio', 'with user_aisle_pacing_ratio'],
    'val_aucpr': [aucpr_without, aucpr_with],
    'best_iteration': [model_without.best_iteration, model_with.best_iteration]})
print(metric_comparison)
print(f"PR-AUC change: {aucpr_with - aucpr_without:+.5f}")

##### Feature importance: gain and mean absolute SHAP for the model with the feature

In [ ]:
gain = pd.Series(model_with.get_booster().get_score(importance_type='gain')).reindex(feature_list).fillna(0)

shap_sample = val_df[feature_list].sample(n=min(5000, len(val_df)), random_state=42)
shap_values = shap.TreeExplainer(model_with).shap_values(shap_sample)
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_list)

importance = pd.DataFrame({'gain': gain, 'mean_abs_shap': mean_abs_shap})
importance['gain_rank'] = importance['gain'].rank(ascending=False).astype(int)
importance['shap_rank'] = importance['mean_abs_shap'].rank(ascending=False).astype(int)
importance = importance.sort_values('gain', ascending=False)
print(importance)
print("\nuser_aisle_pacing_ratio ranks:",
      f"gain #{importance.loc['user_aisle_pacing_ratio', 'gain_rank']},",
      f"SHAP #{importance.loc['user_aisle_pacing_ratio', 'shap_rank']} of {len(feature_list)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
importance['gain'].sort_values().plot.barh(ax=axes[0], title='Gain')
importance['mean_abs_shap'].sort_values().plot.barh(ax=axes[1], title='Mean |SHAP|')
plt.tight_layout()
plt.show()